# Static detection — human validation (33 random samples)

Loads snippets **flagged as static** from `dataset_construction/reports/static_scan_results.jsonl`, draws **33 random** rows (reproducible with `RANDOM_SEED`), and opens a **Gradio** UI to label each clip as **really static** (frozen / slideshow) or **not** (motion the metric missed).

Labels append to `dataset_construction/notebooks/static_human_labels.jsonl` (one JSON object per line). Delete that file if you want a fresh labeling run with the same random sample. Requires `gradio` in your environment.

In [9]:
from __future__ import annotations

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import gradio as gr


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "dataset_construction").is_dir():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find repo root (dataset_construction/).")


REPO_ROOT = find_repo_root()
SCAN_RESULTS = REPO_ROOT / "dataset_construction/reports/static_scan_results.jsonl"
LABELS_OUT = REPO_ROOT / "dataset_construction/notebooks/static_human_labels.jsonl"
RANDOM_SEED = 42
N_SAMPLES = 33

print("REPO_ROOT:", REPO_ROOT)
print("Scan results:", SCAN_RESULTS.exists(), SCAN_RESULTS)

REPO_ROOT: /Users/viktorzozula/Documents/Cats-in-Pain-Bachelors
Scan results: True /Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/dataset_construction/reports/static_scan_results.jsonl


In [10]:
def load_static_rows() -> list[dict]:
    rows: list[dict] = []
    with open(SCAN_RESULTS, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            if r.get("status") != "static":
                continue
            vp = r.get("video_path")
            if not vp or not Path(vp).is_file():
                continue
            rows.append(r)
    return rows


all_static = load_static_rows()
random.seed(RANDOM_SEED)
k = min(N_SAMPLES, len(all_static))
sample = random.sample(all_static, k=k) if all_static else []
print(f"Static rows with valid file on disk: {len(all_static)} | sampled: {len(sample)}")

Static rows with valid file on disk: 1468 | sampled: 33


In [12]:
def build_demo(rows: list[dict]):
    if not rows:
        with gr.Blocks() as demo:
            gr.Markdown("No data. Check `static_scan_results.jsonl`.")
        return demo

    with gr.Blocks(title="Static validation") as demo:
        gr.Markdown(
            "**Really static** = frozen frame / slideshow. **Not** = visible motion (false positive). "
            "Each click appends one line to `dataset_construction/notebooks/static_human_labels.jsonl`."
        )
        idx = gr.State(0)
        vid = gr.Video(label="Snippet")
        info = gr.Markdown()
        really = gr.Radio(choices=["yes", "no"], label="Really static?", value="yes")
        status = gr.Textbox(label="Last action", interactive=False)
        go = gr.Button("Save & next")

        def render(i: int):
            i = max(0, min(int(i), len(rows) - 1))
            r = rows[i]
            md = (
                f"### {i + 1} / {len(rows)}\n"
                f"`{r['snippet_id']}`  \n"
                f"{r.get('platform')} · {r.get('behavioral_category')}  \n\n"
                f"mean_variance: **{r.get('mean_variance')}**"
            )
            return r["video_path"], md

        def init():
            vp, md = render(0)
            # gr.update avoids postprocess errors; repo paths need allowed_paths= on launch
            return 0, gr.update(value=str(vp)), md, "Ready."

        def step(cur, rs):
            cur = int(cur)
            r = rows[cur]
            rec = {
                "snippet_id": r["snippet_id"],
                "human_really_static": rs == "yes",
                "mean_variance": r.get("mean_variance"),
                "platform": r.get("platform"),
                "behavioral_category": r.get("behavioral_category"),
                "labeled_at": datetime.now(timezone.utc).isoformat(),
            }
            LABELS_OUT.parent.mkdir(parents=True, exist_ok=True)
            with open(LABELS_OUT, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            nxt = cur + 1
            if nxt >= len(rows):
                vp, md = render(cur)
                return (
                    cur,
                    gr.update(value=str(vp)),
                    md + "\n\n### ✅ All samples labeled",
                    f"Saved `{r['snippet_id']}` — finished ({len(rows)}/{len(rows)}).",
                )
            vp, md = render(nxt)
            return nxt, gr.update(value=str(vp)), md, f"Saved `{r['snippet_id']}` → now {nxt + 1}/{len(rows)}"

        demo.load(init, outputs=[idx, vid, info, status])
        go.click(step, inputs=[idx, really], outputs=[idx, vid, info, status])

    return demo


demo = build_demo(sample)
demo

Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 outputs:
 |-<gradio.components.state.State object at 0x11669df90>
 |-<gradio.components.video.Video object at 0x11669e0d0>
 |-<gradio.components.markdown.Markdown object at 0x11669e490>
 |-<gradio.components.textbox.Textbox object at 0x11669e710>
fn_index=1
 inputs:
 |-<gradio.components.state.State object at 0x11669df90>
 |-<gradio.components.radio.Radio object at 0x11669e5d0>
 outputs:
 |-<gradio.components.state.State object at 0x11669df90>
 |-<gradio.components.video.Video object at 0x11669e0d0>
 |-<gradio.components.markdown.Markdown object at 0x11669e490>
 |-<gradio.components.textbox.Textbox object at 0x11669e710>

In [13]:
# Run the app (inline in Jupyter / VS Code notebook).
# allowed_paths lets Gradio read snippet videos under the repo (otherwise cache/postprocess errors).
demo.launch(inline=True, share=False, allowed_paths=[str(REPO_ROOT)])

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
